# FORMAT ALL CAMS FILES

In [1]:
import os
import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import xarray as xr
from datetime import datetime, timedelta
import netCDF4

In [2]:
root = "/home/pserrano/data/providentia/exp_to_interp"

In [3]:
input_files = {'cams_analysis_regional': f'{root}/cams_analysis_ensemble/regional/hourly/sconcno2/ENS_ANALYSIS.nc',
 'cams_forecast_regional': f'{root}/cams_forecast_ensemble/regional/hourly/sconcno2/ENS_FORECAST.nc',
 'cams_forecast_global': f'{root}/cams_forecast/global/hourly/sconcno2/data_mlev.nc',
 'cams_reanalysis_regional': f'{root}/cams_reanalysis_ensemble/regional/hourly/sconcno2/cams.eaq.ira.ENSa.no2.l0.2024-12.nc',
 'cams_reanalysis_global': f'{root}/cams_reanalysis/global/hourly/sconcno2/data_mlev.nc'}

In [4]:
output_files = {'cams_analysis_regional': f'{root}/cams_analysis_ensemble/regional/hourly/sconcno2/sconcno2_20250720.nc',
 'cams_forecast_regional': f'{root}/cams_forecast_ensemble/regional/hourly/sconcno2/sconcno2_20250701.nc',
 'cams_forecast_global': f'{root}/cams_forecast/global/hourly/sconcno2/sconcno2_20250701.nc',
 'cams_reanalysis_regional': f'{root}/cams_reanalysis_ensemble/regional/hourly/sconcno2/sconcno2_20250701.nc',
 'cams_reanalysis_global': f'{root}/cams_reanalysis/global/hourly/sconcno2/sconcno2_201802.nc'}

In [5]:
def show_variable(var):
    print(f"• Variable: {var.name}")
    print(f"• Data type: {var.dtype}")
    print(f"• Dimensions: {var.dimensions}")
    print(f"• Shape: {var.shape}")
    print("• Attributes:")
    if var.ncattrs():
        for attr in var.ncattrs():
            print(f"   - {attr}: {getattr(var, attr)}")
    print()

## GOAL

In [55]:
goal_path = output_files['cams_analysis_regional']

In [56]:
with Dataset(goal_path, "r") as goal_file:
    for v in goal_file.variables.keys():
        show_variable(goal_file.variables[v])
        print("------------------------------------------\n")
    print(f"• Dimensions: {list(goal_file.dimensions.keys())}")

• Variable: longitude
• Data type: float32
• Dimensions: ('longitude',)
• Shape: (700,)
• Attributes:

------------------------------------------

• Variable: latitude
• Data type: float32
• Dimensions: ('latitude',)
• Shape: (420,)
• Attributes:

------------------------------------------

• Variable: level
• Data type: float32
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - positive: up

------------------------------------------

• Variable: time
• Data type: float32
• Dimensions: ('time',)
• Shape: (12,)
• Attributes:
   - calendar: standard
   - units: hours since 2025-07-20

------------------------------------------

• Variable: sconcno2
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (12, 1, 420, 700)
• Attributes:
   - coordinates: latitude longitude
   - grid_mapping: crs
   - units: µg/m3

------------------------------------------

• Variable: crs
• Data type: uint8
• Dimensions: ()
• Shape: ()
• Attributes:
   - grid_mapping

## cams_analysis_regional

Open the netcdf file to format and create the new one.

In [8]:
input_path = input_files['cams_analysis_regional']
input_file = Dataset(input_path, "r")

In [9]:
output_path = output_files['cams_analysis_regional']
output_file = Dataset(output_path, "w")

Inspect the original netcdf file

In [10]:
for v in input_file.variables.keys():
    show_variable(input_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(input_file.dimensions.keys())}")

• Variable: longitude
• Data type: float32
• Dimensions: ('longitude',)
• Shape: (700,)
• Attributes:
   - long_name: longitude
   - units: degrees_east

------------------------------------------

• Variable: latitude
• Data type: float32
• Dimensions: ('latitude',)
• Shape: (420,)
• Attributes:
   - long_name: latitude
   - units: degrees_north

------------------------------------------

• Variable: level
• Data type: float32
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - long_name: level
   - units: m

------------------------------------------

• Variable: time
• Data type: float32
• Dimensions: ('time',)
• Shape: (12,)
• Attributes:
   - long_name: ANALYSIS time from 20250720
   - units: hours

------------------------------------------

• Variable: no2_conc
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (12, 1, 420, 700)
• Attributes:
   - _FillValue: -999.0
   - species: Nitrogen Dioxide
   - units: µg/m3
   - value: hourly val

After inspecting the input file, manually insert the name for all the possible variables.

In [11]:
longitude_var_name = 'longitude'
latitude_var_name = 'latitude'
level_var_name = 'level'
time_var_name = 'time'
species_var_name = 'no2_conc'
# -----------------------------
longitude_dim_name = 'longitude'
latitude_dim_name = 'latitude'
level_dim_name = 'level'
time_dim_name = 'time'

Map the original names to the names accepted by Providentia.

In [12]:
cams_providentia_map = {
    time_var_name : 'time', 
    species_var_name : 'sconcno2', 
    longitude_var_name : 'longitude', 
    latitude_var_name : 'latitude',
    level_var_name : 'level'} 
#--------------------------------------
cams_providentia_dim_map = {
     time_dim_name : 'time', 
     longitude_dim_name : 'longitude', 
     latitude_dim_name : 'latitude',
     level_dim_name : 'level'}

Also, manually insert the name for the atributes.

It is mandatory to have the **calendar** and the **units** attribute in the **time** variable.

- Valid **calendars** are ‘standard’, ‘gregorian’, ‘proleptic_gregorian’, ‘noleap’, ‘365_day’, ‘360_day’, ‘julian’, ‘all_leap’, ‘366_day’.
- The correct format for **units** is a string of the form ‘*time units* since’ reference time describing the time units. *time units* can be days, hours, minutes, seconds, milliseconds or microseconds. *reference time* is the time origin.

In [13]:
time_calendar = 'standard'
time_units = 'hours since 2025-07-20'

It is mandatory to have the **coordinates**, the **grid_mapping** and the **units** attribute in the ***species*** variable.

- The **coordinates** format is a string of the form *latitude_name* *longitude_name*.
- Valid **grid_mapping** are ‘crs’, ‘rotated_pole’, ‘Lambert_conformal’, ‘Lambert_Conformal’.
- The possible **units** are in the **load_dictionaries** method of the **UnitConverter** class in the **unit_converter.py** file.

In [14]:
species_coordinates = 'latitude longitude'
species_grid_mapping = 'crs'
species_units = input_file['no2_conc'].units

* The coordinates attribute helps providentia identify the longitude and latitude name in the ncfile. Meaning longitude and latitude can take any names, it just have to be correspond to the *variables* and *dimensions* names.

It is mandatory to have the **positive** attribute in the **level** variable.

- There is only two valid **positive** values: ‘up’ and ‘down’.

In [15]:
level_positive = 'up'

Add the dimensions to the new file.

In [16]:
for dim_name, dim in input_file.dimensions.items():
    # create the dimension with the new name
    output_file.createDimension(cams_providentia_dim_map[dim_name], len(dim))
    print(f"Created {dim_name} dimension")

Created longitude dimension
Created latitude dimension
Created level dimension
Created time dimension


Make sure dimensions of the variable are in the following order: **('time', 'level', 'latitude', 'longitude')**

Providentia uses this order because it follows the **COARDS Conventions** which recommend that spatiotemporal dimensions appear in this specific order: T, then Z, then Y, then X in the CDL definition corresponding to the file. All other dimensions should, whenever possible, be placed to the left of the spatiotemporal dimensions.

In [17]:
input_file['no2_conc'].dimensions

('time', 'level', 'latitude', 'longitude')

We are good in this case.

Add the variables and its attributes.

In [18]:
for input_var_name, input_var in input_file.variables.items():

    # change the name of the input variable into the providentia name
    output_var_name = cams_providentia_map[input_var_name]

    # change the name of the dimensions into the providentia name
    output_var_dims = [cams_providentia_dim_map.get(name, name) for name in input_var.dimensions]

    # create the variable
    var = output_file.createVariable(output_var_name, input_var.datatype, output_var_dims)        

    # add the data to the variable
    var[:] = input_var[:]

    # add calendar and units attributes to the time variable
    if input_var_name == time_var_name:
        var.setncattr('calendar', time_calendar)
        var.setncattr('units', time_units)
    # add coordinates, grid_mapping and units to the species variable
    elif input_var_name == species_var_name:
        var.setncattr('coordinates', species_coordinates)
        var.setncattr('grid_mapping', species_grid_mapping)
        var.setncattr('units', species_units)
    # add positive to level
    elif input_var_name == level_var_name:
        var.setncattr('positive', level_positive)

Add the crs variable to the netcdf file

In [19]:
crs_var = output_file.createVariable('crs', 'u1')  
crs_var.setncatts({
    'grid_mapping_name': 'latitude_longitude',
    'semi_major_axis': 6371000.0,
    'inverse_flattening': 0.0
})

Finally, inspect the output file.

In [20]:
for v in output_file.variables.keys():
    show_variable(output_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(output_file.dimensions.keys())}")

• Variable: longitude
• Data type: float32
• Dimensions: ('longitude',)
• Shape: (700,)
• Attributes:

------------------------------------------

• Variable: latitude
• Data type: float32
• Dimensions: ('latitude',)
• Shape: (420,)
• Attributes:

------------------------------------------

• Variable: level
• Data type: float32
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - positive: up

------------------------------------------

• Variable: time
• Data type: float32
• Dimensions: ('time',)
• Shape: (12,)
• Attributes:
   - calendar: standard
   - units: hours since 2025-07-20

------------------------------------------

• Variable: sconcno2
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (12, 1, 420, 700)
• Attributes:
   - coordinates: latitude longitude
   - grid_mapping: crs
   - units: µg/m3

------------------------------------------

• Variable: crs
• Data type: uint8
• Dimensions: ()
• Shape: ()
• Attributes:
   - grid_mapping

In [21]:
input_file.close()
output_file.close()

## General formating function

In [22]:
def format_file(input_file, output_file, cams_providentia_map, cams_providentia_dim_map):      
    for input_dim_name, output_dim_name in cams_providentia_dim_map.items():
        # get dimension
        dim = input_file.dimensions[input_dim_name]
        # create the dimension with the new name 
        output_file.createDimension(output_dim_name, len(dim))
    
    for input_var_name, output_var_name in cams_providentia_map.items():
        # get the variable
        input_var = input_file[input_var_name]
    
        # change the name of the dimensions into the providentia name
        output_var_dims = [cams_providentia_dim_map.get(name, name) for name in input_var.dimensions if name in cams_providentia_dim_map]
        
        # create the variable
        var = output_file.createVariable(output_var_name, input_var.datatype, output_var_dims)        
    
        # add the data to the variable
        var[:] = input_var[:]
    
        # add calendar and units attributes to the time variable
        if input_var_name == time_var_name:
            var.setncattr('calendar', time_calendar)
            var.setncattr('units', time_units)
            
        # add coordinates, grid_mapping and units to the species variable
        elif input_var_name == species_var_name:
            var.setncattr('coordinates', species_coordinates)
            var.setncattr('grid_mapping', species_grid_mapping)
            var.setncattr('units', species_units)

        # add positive to level
        elif input_var_name == level_var_name:
            var.setncattr('positive', level_positive)

    # add the crs variable to the netcdf file
    crs_var = output_file.createVariable('crs', 'u1')  
    crs_var.setncatts({
        'grid_mapping_name': 'latitude_longitude',
        'semi_major_axis': 6371000.0,
        'inverse_flattening': 0.0
    })

## cams_forecast_regional

In [23]:
input_path = input_files['cams_forecast_regional']
input_file = Dataset(input_path, "r")

output_path = output_files['cams_forecast_regional']
output_file = Dataset(output_path, "w")

In [24]:
for v in input_file.variables.keys():
    show_variable(input_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(input_file.dimensions.keys())}")

• Variable: longitude
• Data type: float32
• Dimensions: ('longitude',)
• Shape: (700,)
• Attributes:
   - long_name: longitude
   - units: degrees_east

------------------------------------------

• Variable: latitude
• Data type: float32
• Dimensions: ('latitude',)
• Shape: (420,)
• Attributes:
   - long_name: latitude
   - units: degrees_north

------------------------------------------

• Variable: level
• Data type: float32
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - long_name: level
   - units: m

------------------------------------------

• Variable: time
• Data type: float32
• Dimensions: ('time',)
• Shape: (97,)
• Attributes:
   - long_name: FORECAST time from 20250701
   - units: hours

------------------------------------------

• Variable: no2_conc
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (97, 1, 420, 700)
• Attributes:
   - _FillValue: -999.0
   - species: Nitrogen Dioxide
   - units: µg/m3
   - value: hourly val

In [25]:
time_calendar = 'standard'
time_units = 'hours since 2025-07-01'
# --------------------------------------
species_coordinates = 'latitude longitude'
species_grid_mapping = 'crs'
species_units = input_file['no2_conc'].units
# --------------------------------------
level_positive = 'up'

In [26]:
longitude_var_name = 'longitude'
latitude_var_name = 'latitude'
level_var_name = 'level'
time_var_name = 'time'
species_var_name = 'no2_conc'
# -----------------------------
longitude_dim_name = 'longitude'
latitude_dim_name = 'latitude'
level_dim_name = 'level'
time_dim_name = 'time'

In [27]:
cams_providentia_map = {
    time_var_name : 'time', 
    species_var_name : 'sconcno2', 
    longitude_var_name : 'longitude', 
    latitude_var_name : 'latitude',
    level_var_name : 'level'} 
# --------------------------------------
cams_providentia_dim_map = {
     time_dim_name : 'time', 
     longitude_dim_name : 'longitude', 
     latitude_dim_name : 'latitude',
     level_dim_name : 'level'}

In [28]:
format_file(input_file, output_file, cams_providentia_map, cams_providentia_dim_map)

In [29]:
for v in output_file.variables.keys():
    show_variable(output_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(output_file.dimensions.keys())}")

• Variable: time
• Data type: float32
• Dimensions: ('time',)
• Shape: (97,)
• Attributes:
   - calendar: standard
   - units: hours since 2025-07-01

------------------------------------------

• Variable: sconcno2
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (97, 1, 420, 700)
• Attributes:
   - coordinates: latitude longitude
   - grid_mapping: crs
   - units: µg/m3

------------------------------------------

• Variable: longitude
• Data type: float32
• Dimensions: ('longitude',)
• Shape: (700,)
• Attributes:

------------------------------------------

• Variable: latitude
• Data type: float32
• Dimensions: ('latitude',)
• Shape: (420,)
• Attributes:

------------------------------------------

• Variable: level
• Data type: float32
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - positive: up

------------------------------------------

• Variable: crs
• Data type: uint8
• Dimensions: ()
• Shape: ()
• Attributes:
   - grid_mapping

In [30]:
input_file.close()
output_file.close()

## cams_forecast_global

In [31]:
input_path = input_files['cams_forecast_global']
input_file = Dataset(input_path, "r")

output_path = output_files['cams_forecast_global']
output_file = Dataset(output_path, "w")

In [32]:
for v in input_file.variables.keys():
    show_variable(input_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(input_file.dimensions.keys())}")

• Variable: forecast_period
• Data type: float64
• Dimensions: ('forecast_period',)
• Shape: (41,)
• Attributes:
   - _FillValue: nan
   - long_name: time since forecast_reference_time
   - standard_name: forecast_period
   - dtype: float64
   - units: hours

------------------------------------------

• Variable: forecast_reference_time
• Data type: int64
• Dimensions: ('forecast_reference_time',)
• Shape: (1,)
• Attributes:
   - long_name: initial time of forecast
   - standard_name: forecast_reference_time
   - units: seconds since 1970-01-01
   - calendar: proleptic_gregorian

------------------------------------------

• Variable: model_level
• Data type: float64
• Dimensions: ('model_level',)
• Shape: (1,)
• Attributes:
   - _FillValue: nan
   - long_name: hybrid level
   - units: 1
   - positive: down
   - standard_name: atmosphere_hybrid_sigma_pressure_coordinate

------------------------------------------

• Variable: latitude
• Data type: float64
• Dimensions: ('latitude',)
•

In [33]:
time_calendar = 'proleptic_gregorian'
time_units = 'hours since 2025-07-01'
#--------------------------------------
species_coordinates = 'latitude longitude'
species_grid_mapping = 'crs'
species_units = input_file['no2'].units
# --------------------------------------
level_positive = 'up'

In [34]:
longitude_var_name = 'longitude'
latitude_var_name = 'latitude'
level_var_name = 'model_level'
time_var_name = 'forecast_period'
species_var_name = 'no2'
# -----------------------------
longitude_dim_name = 'longitude'
latitude_dim_name = 'latitude'
level_dim_name = 'model_level'
time_dim_name = 'forecast_period'

In [35]:
cams_providentia_map = {
    time_var_name : 'time', 
    species_var_name : 'sconcno2', 
    longitude_var_name : 'longitude', 
    latitude_var_name : 'latitude',
    level_var_name : 'level'} 
#--------------------------------------
cams_providentia_dim_map = {
     time_dim_name : 'time', 
     longitude_dim_name : 'longitude', 
     latitude_dim_name : 'latitude',
     level_dim_name : 'level'}

In [36]:
format_file(input_file, output_file, cams_providentia_map, cams_providentia_dim_map)

In [37]:
for v in output_file.variables.keys():
    show_variable(output_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(output_file.dimensions.keys())}")

• Variable: time
• Data type: float64
• Dimensions: ('time',)
• Shape: (41,)
• Attributes:
   - calendar: proleptic_gregorian
   - units: hours since 2025-07-01

------------------------------------------

• Variable: sconcno2
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (41, 1, 451, 900)
• Attributes:
   - coordinates: latitude longitude
   - grid_mapping: crs
   - units: kg kg**-1

------------------------------------------

• Variable: longitude
• Data type: float64
• Dimensions: ('longitude',)
• Shape: (900,)
• Attributes:

------------------------------------------

• Variable: latitude
• Data type: float64
• Dimensions: ('latitude',)
• Shape: (451,)
• Attributes:

------------------------------------------

• Variable: level
• Data type: float64
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - positive: up

------------------------------------------

• Variable: crs
• Data type: uint8
• Dimensions: ()
• Shape: ()
• Attributes:
  

In [38]:
input_file.close()
output_file.close()

## cams_reanalysis_regional

In [39]:
input_path = input_files['cams_forecast_regional']
input_file = Dataset(input_path, "r")

output_path = output_files['cams_forecast_regional']
output_file = Dataset(output_path, "w")

In [40]:
for v in input_file.variables.keys():
    show_variable(input_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(input_file.dimensions.keys())}")

• Variable: longitude
• Data type: float32
• Dimensions: ('longitude',)
• Shape: (700,)
• Attributes:
   - long_name: longitude
   - units: degrees_east

------------------------------------------

• Variable: latitude
• Data type: float32
• Dimensions: ('latitude',)
• Shape: (420,)
• Attributes:
   - long_name: latitude
   - units: degrees_north

------------------------------------------

• Variable: level
• Data type: float32
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - long_name: level
   - units: m

------------------------------------------

• Variable: time
• Data type: float32
• Dimensions: ('time',)
• Shape: (97,)
• Attributes:
   - long_name: FORECAST time from 20250701
   - units: hours

------------------------------------------

• Variable: no2_conc
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (97, 1, 420, 700)
• Attributes:
   - _FillValue: -999.0
   - species: Nitrogen Dioxide
   - units: µg/m3
   - value: hourly val

In [41]:
time_calendar = 'standard'
time_units = 'hours since 2025-07-20'
#--------------------------------------
species_coordinates = 'latitude longitude'
species_grid_mapping = 'crs'
species_units = input_file['no2_conc'].units
# --------------------------------------
level_positive = 'up'

In [42]:
longitude_var_name = 'longitude'
latitude_var_name = 'latitude'
level_var_name = 'level'
time_var_name = 'time'
species_var_name = 'no2_conc'
# -----------------------------
longitude_dim_name = 'longitude'
latitude_dim_name = 'latitude'
level_dim_name = 'level'
time_dim_name = 'time'

In [43]:
cams_providentia_map = {
    time_var_name : 'time', 
    species_var_name : 'sconcno2', 
    longitude_var_name : 'longitude', 
    latitude_var_name : 'latitude',
    level_var_name : 'level'} 
#--------------------------------------
cams_providentia_dim_map = {
     time_dim_name : 'time', 
     longitude_dim_name : 'longitude', 
     latitude_dim_name : 'latitude',
     level_dim_name : 'level'}

In [44]:
format_file(input_file, output_file, cams_providentia_map, cams_providentia_dim_map)

In [45]:
for v in output_file.variables.keys():
    show_variable(output_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(output_file.dimensions.keys())}")

• Variable: time
• Data type: float32
• Dimensions: ('time',)
• Shape: (97,)
• Attributes:
   - calendar: standard
   - units: hours since 2025-07-20

------------------------------------------

• Variable: sconcno2
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (97, 1, 420, 700)
• Attributes:
   - coordinates: latitude longitude
   - grid_mapping: crs
   - units: µg/m3

------------------------------------------

• Variable: longitude
• Data type: float32
• Dimensions: ('longitude',)
• Shape: (700,)
• Attributes:

------------------------------------------

• Variable: latitude
• Data type: float32
• Dimensions: ('latitude',)
• Shape: (420,)
• Attributes:

------------------------------------------

• Variable: level
• Data type: float32
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - positive: up

------------------------------------------

• Variable: crs
• Data type: uint8
• Dimensions: ()
• Shape: ()
• Attributes:
   - grid_mapping

In [46]:
input_file.close()
output_file.close()

## cams_reanalysis_global

In [47]:
input_path = input_files['cams_reanalysis_global']
input_file = Dataset(input_path, "r")

output_path = output_files['cams_reanalysis_global']
output_file = Dataset(output_path, "w")

In [48]:
for v in input_file.variables.keys():
    show_variable(input_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(input_file.dimensions.keys())}")

• Variable: valid_time
• Data type: int64
• Dimensions: ('valid_time',)
• Shape: (2,)
• Attributes:
   - long_name: time
   - standard_name: time
   - units: seconds since 1970-01-01
   - calendar: proleptic_gregorian

------------------------------------------

• Variable: model_level
• Data type: float64
• Dimensions: ('model_level',)
• Shape: (1,)
• Attributes:
   - _FillValue: nan
   - long_name: hybrid level
   - units: 1
   - positive: down
   - standard_name: atmosphere_hybrid_sigma_pressure_coordinate

------------------------------------------

• Variable: latitude
• Data type: float64
• Dimensions: ('latitude',)
• Shape: (241,)
• Attributes:
   - _FillValue: nan
   - units: degrees_north
   - standard_name: latitude
   - long_name: latitude
   - stored_direction: decreasing

------------------------------------------

• Variable: longitude
• Data type: float64
• Dimensions: ('longitude',)
• Shape: (480,)
• Attributes:
   - _FillValue: nan
   - units: degrees_east
   - standar

In [49]:
time_calendar = 'proleptic_gregorian'
time_units = 'hours since 2025-07-20'
#--------------------------------------
species_coordinates = 'latitude longitude'
species_grid_mapping = 'crs'
species_units = input_file['no2'].units
# --------------------------------------
level_positive = 'up'

In [50]:
longitude_var_name = 'longitude'
latitude_var_name = 'latitude'
level_var_name = 'model_level'
time_var_name = 'valid_time'
species_var_name = 'no2'
# -----------------------------
longitude_dim_name = 'longitude'
latitude_dim_name = 'latitude'
level_dim_name = 'model_level'
time_dim_name = 'valid_time'

In [51]:
cams_providentia_map = {
    time_var_name : 'time', 
    species_var_name : 'sconcno2', 
    longitude_var_name : 'longitude', 
    latitude_var_name : 'latitude',
    level_var_name : 'level'} 
#--------------------------------------
cams_providentia_dim_map = {
     time_dim_name : 'time', 
     longitude_dim_name : 'longitude', 
     latitude_dim_name : 'latitude',
     level_dim_name : 'level'}

In [52]:
format_file(input_file, output_file, cams_providentia_map, cams_providentia_dim_map)

In [53]:
for v in output_file.variables.keys():
    show_variable(output_file.variables[v])
    print("------------------------------------------\n")
print(f"• Dimensions: {list(output_file.dimensions.keys())}")

• Variable: time
• Data type: int64
• Dimensions: ('time',)
• Shape: (2,)
• Attributes:
   - calendar: proleptic_gregorian
   - units: hours since 2025-07-20

------------------------------------------

• Variable: sconcno2
• Data type: float32
• Dimensions: ('time', 'level', 'latitude', 'longitude')
• Shape: (2, 1, 241, 480)
• Attributes:
   - coordinates: latitude longitude
   - grid_mapping: crs
   - units: kg kg**-1

------------------------------------------

• Variable: longitude
• Data type: float64
• Dimensions: ('longitude',)
• Shape: (480,)
• Attributes:

------------------------------------------

• Variable: latitude
• Data type: float64
• Dimensions: ('latitude',)
• Shape: (241,)
• Attributes:

------------------------------------------

• Variable: level
• Data type: float64
• Dimensions: ('level',)
• Shape: (1,)
• Attributes:
   - positive: up

------------------------------------------

• Variable: crs
• Data type: uint8
• Dimensions: ()
• Shape: ()
• Attributes:
   - g

In [54]:
input_file.close()
output_file.close()